In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType
from datetime import datetime, timedelta
import random

# Esquema da tabela bronze
clientes_schema = StructType([
    StructField('cliente_id', IntegerType(), False),
    StructField('nome', StringType(), True),
    StructField('email', StringType(), True),
    StructField('data_nascimento', DateType(), True),
    StructField('pais', StringType(), True),
    StructField('renda', DoubleType(), True),
    StructField('investimento', DoubleType(), True),
    StructField('idade', IntegerType(), True)
])

# Gerar dados fake
nomes = [
    'Ana', 'Bruno', 'Carlos', 'Daniela', 'Eduardo', 'Fernanda', 'Gabriel', 'Helena', 'Igor', 'Juliana',
    'Kleber', 'Larissa', 'Marcos', 'Natália', 'Otávio', 'Patrícia', 'Quintino', 'Renata', 'Samuel', 'Tatiane',
    'Ulisses', 'Valéria', 'Wagner', 'Xuxa', 'Yasmin'
]
paises = ['Brasil', 'Argentina', 'Chile', 'Uruguai', 'Paraguai']

clientes_data = []
for i in range(1, 26):
    nome = nomes[i-1]
    email = f"{nome.lower()}@exemplo.com"
    data_nascimento = datetime(1980, 1, 1) + timedelta(days=random.randint(0, 15000))
    pais = random.choice(paises)
    renda = round(random.uniform(2000, 20000), 2)
    investimento = round(random.uniform(0, 100000), 2)
    idade = int((datetime.now() - data_nascimento).days // 365)
    clientes_data.append((i, nome, email, data_nascimento, pais, renda, investimento, idade))

clientes_df = spark.createDataFrame(clientes_data, schema=clientes_schema)

# Salvar como tabela bronze com mergeSchema
clientes_df.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.clientes")

display(clientes_df)

In [0]:
from pyspark.sql.functions import *

REGRA 1 SEGMENTAÇÃO PRIME (IDADE SUPERIOR A 35 anos) - (RENDA SUPERIOR A  R$10.000) - (INVESTIMENTO SUPERIOR A R$50.000)

In [0]:
%sql
SELECT *,
  CASE
    WHEN idade > 35 AND investimento > 50000 AND renda > 10000 THEN 'prime'
    ELSE 'outros'
  END AS segmento_prime
FROM bronze.clientes

In [0]:
clientes_df = clientes_df.withColumn(
    "seg_principal",
    when(
        (col("investimento") > 40000) & (col("renda") > 5000),
        "principal"
    ).otherwise("outros")
)

display(clientes_df)

In [0]:
clientes_df = clientes_df.drop("segmento")


In [0]:
%sql
SELECT *,
  CASE
    WHEN idade > 35 AND investimento > 50000 AND renda > 10000 THEN 'prime'
    ELSE 'outros'
  END AS segmento_prime
FROM bronze.clientes